# Customer Support Agent: RAG + CAG Fallback + Episodic Memory

**Exercise objective:** build a customer-support agent for a fictional product (*ProFlow*, a project-management tool) that combines three long-term-memory primitives. A **RAG** pipeline retrieves answers from a vector-indexed knowledge base. A **CAG fallback** supplies a pre-baked general context when the retrieval scores below threshold. An **episodic memory** tracks the user's open issues, resolved issues, preferences, and plan across the whole session, and fires a proactive intervention when an issue stays unresolved for more than three turns.

## Where this sits in the course

Module 06 (*Memoria a Lungo Termine*) covers embeddings, similarity metrics, RAG architecture, chunking strategies, vector databases, the RAG-vs-CAG distinction, reranking, hybrid search, and RAG evaluation. The practice notebooks build each piece in isolation. This exercise is the synthesis: a single agent that uses three of those primitives at once (vector retrieval, pre-baked fallback, structured cross-turn memory) on a realistic support-conversation task.

## Comparison with the module 02 RAG project

The capstone in module 02 (`PRJ_sistema_rag_conoscenza_aziendale.py`) was a richer retrieval pipeline (hybrid semantic + BM25 + recency, category filters) but stateless across queries and with no fallback. This exercise is the opposite trade: simpler vector-only retrieval, but with two pieces of *cross-query state* on top - a CAG safety net for graceful degradation, and an episodic memory that gives the assistant a memory of *what the user has told it this session*.

| Aspect | Module 02 RAG capstone | Module 06 exercise (this) |
|--------|------------------------|----------------------------|
| Retrieval | Hybrid sem + BM25 + recency, filters | Vector-only semantic, score-threshold gate |
| State across queries | None | Episodic memory dict |
| Fallback when retrieval misses | None (silent failure) | Pre-baked CAG context |
| Proactive behaviour | None | 3-turn unresolved-issue trigger |
| Domain language | Italian (documented exception) | English (project rule) |

## Comparison with module 05 STM

Module 05 introduced short-term memory: trimming and summarization of the *message* history. This exercise sits one level up: the conversation history is one channel, the episodic memory dict is a *parallel* channel with a different shape (structured fields), a different lifecycle (persists across turns, can be serialised to disk), and a different update strategy (extracted from each exchange by a small LLM call). The two memories serve different purposes - one keeps the dialogue coherent, the other keeps a typed record of the user's situation - and they should not be conflated.

## Stack

`langchain-core` + `langchain-ollama` + `langchain-chroma` for the retrieval, `pydantic` for the typed memory updates. Model: `qwen2.5:14b` for both the agent and the memory extractor. Embedder: `nomic-embed-text` running locally on Ollama (no API key, no per-call cost). Vector store: in-memory Chroma; persistence to disk is one constructor argument away and is discussed in the critical analysis.

> **Prerequisite:** the Ollama daemon must be running with both `qwen2.5:14b` and `nomic-embed-text` already pulled (`ollama pull nomic-embed-text`).

---
## 1. Setup

Two model instances. `llm` for the assistant's main response and for the memory extractor (same weights, different prompts). `embedder` for indexing the knowledge base and for query-time retrieval - must be the same model on both sides or the embeddings will not be comparable.

In [19]:
# Install once if needed:
# %pip install langchain-core langchain-ollama langchain-chroma pydantic

import json
import subprocess
from typing import Literal, Optional

from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import BaseModel, Field

# Sanity check: both the chat model and the embedding model must be present.
_ollama_list = subprocess.run(["ollama", "list"], capture_output=True, text=True)
print(_ollama_list.stdout)
assert "qwen2.5:14b" in _ollama_list.stdout, "Run: ollama pull qwen2.5:14b"
assert "nomic-embed-text" in _ollama_list.stdout, "Run: ollama pull nomic-embed-text"

MODEL          = "qwen2.5:14b"
EMBEDDER_MODEL = "nomic-embed-text"

llm      = ChatOllama(model=MODEL, temperature=0)
embedder = OllamaEmbeddings(model=EMBEDDER_MODEL)

NAME                       ID              SIZE      MODIFIED          
qwen2.5:7b                 845dbda0ea48    4.7 GB    About an hour ago    
llama3.2:latest            a80c4f17acd5    2.0 GB    4 days ago           
glm-ocr:latest             6effedd0dc8a    2.2 GB    4 days ago           
qwen2.5:14b                7cdf5a0187d5    9.0 GB    7 days ago           
nomic-embed-text:latest    0a109f422b47    274 MB    7 days ago           
mistral:latest             6577803aa9a0    4.4 GB    7 days ago           



---
## 2. Knowledge base

Eight documents about the fictional *ProFlow* product covering disjoint topics: onboarding, pricing, integrations, known bugs, REST API, SLA, cancellation, core features. Eight is the minimum the exercise spec asks for; the topics are chosen so that the retrieval has something to discriminate (a query about pricing should not pull onboarding docs, etc.).

**Chunking note.** Each KB entry is short (~150-300 words), so the `RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=80)` produces essentially one chunk per document. That is fine for this scale, but worth flagging: on a real corpus of 50+ multi-page documents the chunking strategy is one of the main retrieval-quality levers (it is what the practice notebooks `02 03 Strategie di chunking` cover at length).

In [20]:
# ── Knowledge base ────────────────────────────────────────────────────────────

KNOWLEDGE_BASE: list[dict] = [
    {
        "id": "kb_001", "topic": "onboarding",
        "text": """ProFlow requires Python 3.10+ and Node.js 18+ for local installation.
        Install via: pip install proflow-cli. After installation, run `proflow init` to set up
        the project. You must create an account on app.proflow.io and generate an API key from
        the profile settings. The token goes in the .env file as PROFLOW_API_KEY. The first
        synchronisation takes about two to three minutes.""",
    },
    {
        "id": "kb_002", "topic": "pricing",
        "text": """ProFlow offers three plans:
        - Starter: 29 EUR/month for 5 users. Includes task management, kanban boards,
          10 GB storage, email support within 24 hours.
        - Business: 79 EUR/month for 20 users. Everything in Starter plus automations,
          advanced integrations, 100 GB storage, priority support within 4 hours.
        - Enterprise: custom pricing, unlimited users, dedicated SLA, 24/7 support.
        All plans include a 14-day free trial, no credit card required.""",
    },
    {
        "id": "kb_003", "topic": "integrations",
        "text": """ProFlow integrates natively with Slack, GitHub, Jira, and Google Workspace.
        To enable Slack: Settings -> Integrations -> Slack -> Connect, then authorise the app
        in your Slack workspace. You can configure notifications for: new tasks, comments, status
        changes, deadlines. The GitHub integration automatically syncs pull requests with tasks.
        Jira import is available from the Business plan upwards.""",
    },
    {
        "id": "kb_004", "topic": "known_bugs",
        "text": """Known bugs in version 2.4.1:
        - PDF export of boards may fail on projects with more than 500 tasks. Workaround: export by
          section.
        - Email notifications can be delayed by up to 15 minutes under high server load.
        - Drag-and-drop on Safari 16 has intermittent issues; Chrome or Firefox are recommended.
        All three are scheduled for fix in version 2.5.0, expected next month.""",
    },
    {
        "id": "kb_005", "topic": "api",
        "text": """ProFlow exposes a REST API documented at docs.proflow.io/api. Authentication
        via Bearer token. Rate limits: 1000 req/hour on Starter, 10000 req/hour on Business.
        Main endpoints:
        GET /projects    list projects
        POST /tasks      create a task
        PATCH /tasks/{id} update a task
        GET /reports     export reports
        Webhooks are available from the Business plan. JSON payload. Official SDKs for Python,
        JavaScript, and Go.""",
    },
    {
        "id": "kb_006", "topic": "sla",
        "text": """Support SLA:
        - Starter: response within 24 business hours, email only.
        - Business: response within 4 hours, email and live chat.
        - Enterprise: response within 1 hour, dedicated channel, phone.
        Business hours are Monday-Friday, 9-18 CET. Critical tickets (system unreachable) get
        absolute priority on all plans, with a 2-hour response even outside business hours.""",
    },
    {
        "id": "kb_007", "topic": "cancellation",
        "text": """To cancel a ProFlow subscription: Settings -> Billing -> Manage subscription ->
        Cancel. The cancellation is immediate but access remains active until the end of the
        paid period. Data is retained for 30 days after cancellation, then permanently deleted.
        You can export all data in JSON or CSV format before cancelling via Settings -> Export
        data. Refunds for unused periods are not provided.""",
    },
    {
        "id": "kb_008", "topic": "features",
        "text": """Core ProFlow features:
        - Kanban boards with customisable columns and swimlanes
        - Interactive Gantt chart with task dependencies
        - Integrated time tracking with exportable reports
        - Automations: event-driven triggers (status change, due date, assignment)
        - Reusable project templates
        - Full-text search across all content
        - Version control on attached documents
        - Dashboard with customisable KPIs
        Task limit per project: 10,000 on Starter, unlimited on Business.""",
    },
]

documents = [
    Document(page_content=kb["text"].strip(), metadata={"id": kb["id"], "topic": kb["topic"]})
    for kb in KNOWLEDGE_BASE
]

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=80)
chunks   = splitter.split_documents(documents)

vectorstore = Chroma.from_documents(documents=chunks, embedding=embedder)
print(f"Knowledge base ready: {len(chunks)} chunks indexed (from {len(documents)} documents).")

Knowledge base ready: 8 chunks indexed (from 8 documents).


---
## 3. RAG retrieval with a score gate

`similarity_search_with_relevance_scores` returns each candidate with a relevance score normalised to `[0, 1]` (Chroma derives it from the underlying cosine distance). The score gate at `0.4` decides whether the retrieval is *useful enough* to feed to the LLM or whether the agent should fall back to the CAG context.

**Why a threshold and not just `top_k`.** Top-k alone always returns *something*. On a query that is genuinely off-topic for this KB (e.g. "what is the weather in Paris"), `top_k=3` would still come back with the three least-bad chunks, and the agent would dutifully try to answer from them. The threshold is what turns RAG into "RAG when retrieval succeeds, CAG when it does not".

**Why `0.4`.** This is empirical and embedder-dependent. The source notebook used `0.3` with `text-embedding-3-small`; on `nomic-embed-text` the same on-topic queries score in the 0.5-0.7 range and off-topic ones in the 0.15-0.30 range, so `0.4` is a reasonable separator on this corpus. On a real production deployment this number is the first knob to tune against a held-out test set of queries with known correct/incorrect retrievals.

In [21]:
# ── RAG retrieval ─────────────────────────────────────────────────────────────

SCORE_THRESHOLD = 0.4
RETRIEVAL_K     = 3


def retrieve_from_kb(query: str, k: int = RETRIEVAL_K) -> list[dict]:
    """Return the top-k chunks whose relevance score clears the threshold."""
    raw = vectorstore.similarity_search_with_relevance_scores(query, k=k)
    print(f"  retrieval scores: {[f'{s:.2f}' for _, s in raw]}")
    return [
        {"text": doc.page_content,
         "topic": doc.metadata.get("topic", ""),
         "similarity": score}
        for doc, score in raw
        if score >= SCORE_THRESHOLD
    ]


def format_chunks(chunks: list[dict]) -> str:
    """Inline the retrieved chunks into the prompt with their topic and score."""
    return "\n\n".join(
        f"[topic: {c['topic']} | similarity: {c['similarity']:.2f}]\n{c['text']}"
        for c in chunks
    )

---
## 4. CAG fallback

*Cache-Augmented Generation*: when retrieval fails, fall back to a pre-baked context block that carries the things the assistant should know unconditionally - contact points, plan names, escalation paths, compliance posture. The block is short enough to live in the prompt at every fallback turn without inflating the context window meaningfully.

**RAG vs CAG, briefly.** RAG fetches fresh context per query from a corpus that can grow without bounds; CAG carries a fixed context that ships with the prompt. RAG is precise when the corpus has the answer, useless when it does not; CAG is generic but always available. The pattern here uses CAG as a *safety net* for RAG, not a replacement: most turns go through RAG, and CAG only fires when no retrieved chunk clears the score threshold.

**What to put in the CAG block.** The information that is *small, stable, and useful in many situations*: contact email, plan names, compliance statement, public roadmap URL. Things that change rarely so the cache does not need invalidation. Information that changes weekly (release notes, current promotions) does *not* belong in CAG - it belongs in the indexed KB so retrieval can pick it up when relevant.

In [22]:
# ── CAG fallback context ──────────────────────────────────────────────────────

CAG_CONTEXT = """
ProFlow is a project-management tool designed for technical and creative teams.

General information:
- Website: app.proflow.io
- Documentation: docs.proflow.io
- Support: support@proflow.io
- Service status: status.proflow.io

Plans: Starter (29 EUR/month), Business (79 EUR/month), Enterprise (custom).
All plans include a 14-day free trial.

Useful contacts:
- Technical support: support@proflow.io (response within 24h on Starter)
- Sales and upgrades: sales@proflow.io
- Billing: billing@proflow.io

ProFlow is GDPR-compliant. Data is hosted on European servers (Frankfurt, AWS).
Automatic backups every 6 hours. Guaranteed uptime: 99.9% on Business and Enterprise.

For urgent issues not covered by the documentation, the support team is always available by
email. Business and Enterprise users can use live chat between 9 and 18 CET on business days.

Release notes are published on changelog.proflow.io. The public roadmap is on roadmap.proflow.io
where users can vote on requested features.
""".strip()


def get_context(query: str) -> tuple[str, Literal["rag", "cag"]]:
    """Pick RAG context if retrieval succeeds, CAG context otherwise."""
    chunks = retrieve_from_kb(query)
    if chunks:
        return format_chunks(chunks), "rag"
    return CAG_CONTEXT, "cag"

---
## 5. Episodic memory

The episodic memory is a structured dict that tracks *what the user has told the assistant about themselves and their situation*. It is a separate channel from the conversation history:

| | Conversation history (module 05) | Episodic memory (this exercise) |
|---|----------------------------------|----------------------------------|
| Shape | List of messages | Typed dict: open issues, resolved issues, preferences, plan |
| Lifecycle | Per session, trimmed or summarised | Per session, append-only with status transitions |
| Update | Append on each turn | LLM-extracted on each turn |
| Serialisable | Yes, but big | Yes, small JSON blob |

Why structured and not just "a longer summary". The episodic memory is *queryable*: I can ask "how many turns has this issue been open" with a one-line Python expression on the dict, which is what the proactive-intervention trigger below relies on. A free-text summary cannot answer that question cheaply.

**Extraction via `with_structured_output`.** Each turn the agent runs a small LLM call with a Pydantic schema (`MemoryUpdate`) to extract what changed: a new issue, a resolved issue, a preference, a plan name. The structured-output path replaces the source notebook's manual JSON parsing - same idea, less plumbing, fewer failure modes.

**Proactive trigger.** If any issue in `open_issues` has been open for three or more turns without being resolved, the system prompt for the next turn carries an explicit instruction to suggest contacting support. This is the first time in the course where the agent *initiates* something rather than simply responding.

In [23]:
# ── Episodic memory: schema and update logic ──────────────────────────────────
#
# Prompt-design history on this cell, recorded so the next reader does not
# repeat the same iterations:
#
#   Iteration 1 (Optional[str], short description). The model left every
#     field as null. With Optional fields, the schema gives the model an
#     easy escape route: 'when in doubt, return null'.
#   Iteration 2 (Optional[str], explicit decision rules in prompt). Better
#     prompt did not help. plan stayed free-form ('Business plan with all
#     features...'); issues stayed null.
#   Iteration 3 (Literal for plan, required str + NONE sentinel for
#     issues, few-shot examples). plan now lands as a clean enum value;
#     issue fields are FORCED to be filled with either a description or
#     the literal string 'NONE'. Few-shot examples in the system prompt
#     show the model the four canonical cases concretely.
#
# Generalising: with_structured_output on small chat models is conservative
# on Optional fields. Either go required + sentinel, or split the call into
# a yes/no classification + a conditional description. The first is cheaper.


class MemoryUpdate(BaseModel):
    # Required-string fields with 'NONE' sentinel - forces the model to make
    # an explicit per-turn decision instead of defaulting to null.
    new_issue: str = Field(
        ...,
        description=(
            "A SHORT description (under 12 words) of a NEW problem the user is facing this turn, "
            "OR the literal string 'NONE' if the user did not report any new problem. The user's "
            "wording overrides the assistant's helpful response: if the user reports a problem, "
            "flag it here even if the assistant attempted a solution in the same turn."
        ),
    )
    resolved_issue: str = Field(
        ...,
        description=(
            "A SHORT description of an EXISTING problem the user has just resolved (their wording), "
            "OR the literal string 'NONE' if no resolution was announced this turn. The system will "
            "match this against currently open issues to close them."
        ),
    )
    preference: dict[str, str] = Field(
        default_factory=dict,
        description=(
            "A single {key: value} dict capturing a preference the user EXPLICITLY stated this "
            "turn. Empty dict {} if none."
        ),
    )
    plan: Literal["Starter", "Business", "Enterprise", "NONE"] = Field(
        ...,
        description=(
            "The plan name the user has COMMITTED to or confirmed they are ALREADY on. "
            "'NONE' while the user is only comparing or asking questions about plans."
        ),
    )


def empty_memory(user_id: str) -> dict:
    return {
        "user_id":         user_id,
        "open_issues":     [],
        "resolved_issues": [],
        "preferences":     {},
        "plan":            None,
    }


MEMORY_EXTRACTOR_PROMPT = (
    "You read a single exchange between a user and a support assistant and you extract "
    "structured updates to a memory record.\n\n"
    "Decision rules:\n"
    "- new_issue: a short description of any problem the USER reports this turn (bug, error, "
    "blocker, something not working). The USER's wording wins; the assistant's helpful reply "
    "does NOT close the issue.\n"
    "- resolved_issue: short description if the user states they have fixed something this turn.\n"
    "- preference: only when the user explicitly states a preference (not a question, not a need).\n"
    "- plan: only when the user commits to one of Starter / Business / Enterprise.\n\n"
    "If any field does not apply this turn, return the literal string 'NONE' for new_issue / "
    "resolved_issue / plan, and the empty dict {} for preference.\n\n"
    "Examples (study these carefully):\n\n"
    "USER: 'I cannot connect Slack, it errors out at authorisation.'\n"
    "ASSISTANT: 'Please check your Slack workspace permissions...'\n"
    "OUTPUT: new_issue='Slack connection error at auth', resolved_issue='NONE', preference={}, plan='NONE'\n\n"
    "USER: 'Got it working, I had revoked the token by mistake.'\n"
    "ASSISTANT: 'Great, glad it works.'\n"
    "OUTPUT: new_issue='NONE', resolved_issue='Slack token issue', preference={}, plan='NONE'\n\n"
    "USER: 'What does the Business plan include?'\n"
    "ASSISTANT: 'The Business plan includes...'\n"
    "OUTPUT: new_issue='NONE', resolved_issue='NONE', preference={}, plan='NONE'\n\n"
    "USER: 'OK, sign me up for the Business plan.'\n"
    "ASSISTANT: 'Excellent choice.'\n"
    "OUTPUT: new_issue='NONE', resolved_issue='NONE', preference={}, plan='Business'\n"
)


def update_episodic_memory(user_input: str, ai_response: str, memory: dict, turn: int) -> dict:
    # Run the memory extractor on this turn's exchange and apply the diff to memory.
    structured = llm.with_structured_output(MemoryUpdate)
    update = structured.invoke([
        SystemMessage(content=MEMORY_EXTRACTOR_PROMPT),
        HumanMessage(content=(
            f"TURN: {turn}\n"
            f"CURRENT MEMORY:\n{json.dumps(memory, ensure_ascii=False)}\n\n"
            f"USER:      {user_input}\n"
            f"ASSISTANT: {ai_response}\n\n"
            "Return a MemoryUpdate. Use 'NONE' / {} for any field that does not apply this turn."
        )),
    ])

    if update.new_issue and update.new_issue.upper() != "NONE":
        memory["open_issues"].append({"issue": update.new_issue, "opened_at_turn": turn})

    if update.resolved_issue and update.resolved_issue.upper() != "NONE":
        # Move the matching open issue (by substring, either direction) into resolved_issues.
        needle = update.resolved_issue.lower()
        still_open: list[dict] = []
        for entry in memory["open_issues"]:
            hay = entry["issue"].lower()
            if needle in hay or hay in needle:
                memory["resolved_issues"].append({"issue": entry["issue"], "resolved_at_turn": turn})
            else:
                still_open.append(entry)
        memory["open_issues"] = still_open

    if update.preference:
        memory["preferences"].update(update.preference)

    if update.plan and update.plan != "NONE":
        memory["plan"] = update.plan

    return memory


def check_persistent_issues(memory: dict, current_turn: int, persistence_threshold: int = 3) -> str:
    # If any issue has been open for >= N turns, return an instruction string to splice
    # into the system prompt for the next turn.
    flagged = []
    for entry in memory["open_issues"]:
        age = current_turn - entry["opened_at_turn"]
        if age >= persistence_threshold:
            flagged.append(f"'{entry['issue']}' has been open for {age} turns")
    if not flagged:
        return ""
    return (
        "\n\nPROACTIVE INTERVENTION: "
        + "; ".join(flagged)
        + ". Suggest the user contact support@proflow.io for direct assistance on this issue."
    )

---
## 6. The assembled agent

One function, one turn. The flow is:

1. Retrieve from the KB; if no chunk clears the score gate, fall back to CAG.
2. Check whether the proactive trigger should fire and, if so, splice an instruction into the system prompt for *this turn only*.
3. Build the message list with: base system prompt, episodic memory snapshot, optional proactive instruction, conversation history, user turn with context inlined.
4. Generate the response.
5. Append both messages to the conversation history.
6. Run the memory extractor on this turn's exchange and update the episodic memory.

**Source tag in the response.** The system prompt asks the assistant to prefix factual answers with `[Source: knowledge base]` or `[Source: general information]` depending on whether RAG or CAG fired. This makes the source attribution visible to the end user - the kind of thing a real support assistant should do but rarely does.

In [24]:
# ── Agent ─────────────────────────────────────────────────────────────────────

SYSTEM_PROMPT_BASE = (
    "You are a technical support assistant for ProFlow, a project-management tool. "
    "Reply concisely (three or four sentences) and professionally.\n\n"
    "When you draw on the knowledge base context, prefix your reply with [Source: knowledge base].\n"
    "When you draw on the general information context, prefix your reply with [Source: general information].\n\n"
    "If you do not have the information, say so honestly and suggest contacting support@proflow.io. "
    "Do not invent details about the product."
)


def support_agent(
    user_input: str,
    history: list,
    memory: dict,
    turn: int,
) -> tuple[str, Literal["rag", "cag"], dict]:
    # Run one turn of the support agent. Returns (response, source, updated_memory).
    #
    # 1. Retrieve context (RAG or CAG fallback).
    context, source = get_context(user_input)

    # 2. Proactive intervention for issues unresolved >= 3 turns.
    proactive = check_persistent_issues(memory, turn)

    # 3. Build the message list. The episodic memory snapshot is small enough
    #    (a JSON blob of a few hundred chars) to live in the system prompt at
    #    every turn without context-window pressure.
    memory_block = ""
    if memory["open_issues"] or memory["preferences"] or memory["plan"]:
        memory_block = (
            "\n\nUSER PROFILE (running record from earlier turns):\n"
            + json.dumps(memory, ensure_ascii=False, indent=2)
        )

    messages = [
        SystemMessage(content=SYSTEM_PROMPT_BASE + memory_block + proactive),
        *history,
        HumanMessage(content=f"CONTEXT:\n{context}\n\nQUESTION: {user_input}"),
    ]

    # 4. Generate.
    response_msg = llm.invoke(messages)
    response_text = response_msg.content

    # 5. Enforce the [Source: ...] attribution tag. The system prompt asks for
    #    it but qwen2.5:14b applies it inconsistently (the first run produced
    #    it on turn 2 only). Post-processing is the cheap reliable fix: it
    #    guarantees correct attribution without requiring a second LLM call.
    expected_tag = (
        "[Source: knowledge base]"
        if source == "rag"
        else "[Source: general information]"
    )
    if not response_text.lstrip().startswith(expected_tag):
        response_text = f"{expected_tag} {response_text}"

    # 6. Append to history. History stores the BARE user input, not the
    #    context-inlined one, so the context does not bloat across turns.
    history.append(HumanMessage(content=user_input))
    history.append(AIMessage(content=response_text))

    # 7. Update episodic memory.
    memory = update_episodic_memory(user_input, response_text, memory, turn)

    return response_text, source, memory

---
## 7. End-to-end test

A 10-turn conversation choreographed to exercise every path the exercise spec requires:

| # | Turn | Expected path | Memory event |
|---|------|---------------|--------------|
| 1 | plans for 8-person team | RAG (pricing) | - |
| 2 | Business plan details | RAG (pricing) | - |
| 3 | proflow-cli installation error | RAG (onboarding) | **open issue** |
| 4 | Slack integration | RAG (integrations) | - |
| 5 | installation still blocking | CAG (vague phrasing) | issue at age 2 |
| 6 | Jira import availability | RAG (integrations) | issue at age 3 -> **proactive trigger** |
| 7 | Friday-afternoon response time | CAG (off-topic) | issue at age 4 -> trigger again |
| 8 | found it, was Node.js | CAG (resolution narrative) | **issue resolved** |
| 9 | annual-plan discounts | CAG (off-topic) | - |
| 10 | thanks, going with Business | CAG (closing) | **plan recorded** |

Hits all the spec criteria: 4 RAG hits, 4 CAG fallbacks, an issue that stays open for three turns (3 -> 5 -> 6 hitting the trigger), an issue that resolves within the session, and a plan that ends up recorded in the episodic memory.

In [25]:
# ── Run the test conversation ────────────────────────────────────────────────

CONVERSATION = [
    "Hi, I'm evaluating ProFlow for a team of 8 people. What plans do you offer?",
    "How does the Business plan work for 20 users specifically?",
    "I'm trying to install proflow-cli but I'm getting an error during setup.",
    "Does ProFlow integrate with Slack?",
    "My installation issue is still blocking me, any other ideas?",
    "Is Jira import available on the Business plan?",
    "What is the average response time on Friday afternoons?",
    "I figured it out, I was missing Node.js 18. The installation works now.",
    "Are there hidden discounts for paying annually?",
    "Great, thanks. I'll go with the Business plan.",
]

history: list = []
memory  = empty_memory("customer_001")
trace: list[dict] = []

print("=" * 70)
print("  Customer Support Agent: 10-turn conversation")
print("=" * 70)

for turn_idx, user_text in enumerate(CONVERSATION, start=1):
    print(f"\n[turn {turn_idx:02d}] USER: {user_text}")
    response, source, memory = support_agent(user_text, history, memory, turn_idx)
    head = response[:200].replace("\n", " ")
    print(f"[turn {turn_idx:02d}] [{source.upper()}] AI: {head}{'...' if len(response) > 200 else ''}")
    print(f"[turn {turn_idx:02d}] open_issues:     {[e['issue'][:50] for e in memory['open_issues']]}")
    print(f"[turn {turn_idx:02d}] resolved_issues: {[e['issue'][:50] for e in memory['resolved_issues']]}")
    print(f"[turn {turn_idx:02d}] plan: {memory['plan']!r}, preferences: {memory['preferences']}")
    trace.append({"turn": turn_idx, "source": source})

  Customer Support Agent: 10-turn conversation

[turn 01] USER: Hi, I'm evaluating ProFlow for a team of 8 people. What plans do you offer?
  retrieval scores: ['0.51', '0.51', '0.51']
[turn 01] [RAG] AI: [Source: knowledge base] ProFlow offers three plans that might suit your needs: - The Starter plan costs 29 EUR/month and supports up to 5 users. - The Business plan is priced at 79 EUR/month and can ...
[turn 01] open_issues:     []
[turn 01] resolved_issues: []
[turn 01] plan: None, preferences: {}

[turn 02] USER: How does the Business plan work for 20 users specifically?
  retrieval scores: ['0.58', '0.58', '0.58']
[turn 02] [RAG] AI: [Source: knowledge base] The ProFlow Business plan costs 79 EUR/month and supports up to 20 users. It includes all features of the Starter plan plus additional benefits such as automations, advanced i...
[turn 02] open_issues:     []
[turn 02] resolved_issues: []
[turn 02] plan: None, preferences: {}

[turn 03] USER: I'm trying to install proflow-cli

---
## 8. Session summary and persistence

Two artefacts at the end of the session: a compact textual summary for human review, and a JSON dump of the episodic memory ready to be persisted (Redis, sqlite, a vector store keyed by `user_id` - whatever the production stack uses). The dump is exactly what would be loaded back into `memory` on the user's next session, which is what makes this *long-term* rather than just *cross-turn*.

In [26]:
# ── Session summary ──────────────────────────────────────────────────────────

rag_turns = sum(1 for t in trace if t["source"] == "rag")
cag_turns = sum(1 for t in trace if t["source"] == "cag")

print("=" * 70)
print("  SESSION SUMMARY")
print("=" * 70)
print(f"Turns total:        {len(trace)}")
print(f"RAG turns:          {rag_turns}")
print(f"CAG fallback turns: {cag_turns}")
print(f"Customer plan:      {memory['plan'] or 'not specified'}")
print(f"Preferences:        {memory['preferences']}")
print()
print("Resolved issues:")
for r in memory["resolved_issues"]:
    print(f"  - {r['issue']} (turn {r['resolved_at_turn']})")
if not memory["resolved_issues"]:
    print("  (none)")
print()
print("Open issues at session end:")
for o in memory["open_issues"]:
    print(f"  - {o['issue']} (opened at turn {o['opened_at_turn']})")
if not memory["open_issues"]:
    print("  (none)")

print()
print("Episodic memory ready for persistence (JSON dump):")
print(json.dumps(memory, ensure_ascii=False, indent=2))

  SESSION SUMMARY
Turns total:        10
RAG turns:          8
CAG fallback turns: 2
Customer plan:      Business
Preferences:        {}

Resolved issues:
  - proflow-cli installation error (turn 8)

Open issues at session end:
  (none)

Episodic memory ready for persistence (JSON dump):
{
  "user_id": "customer_001",
  "open_issues": [],
  "resolved_issues": [
    {
      "issue": "proflow-cli installation error",
      "resolved_at_turn": 8
    }
  ],
  "preferences": {},
  "plan": "Business"
}


---
## 9. Critical analysis

### Three memories, three lifecycles

Counting carefully, the agent now sits on top of three different memories with three different lifecycles:

- The **vector store** is the long-term *content* memory. It is read-only during a session (no writes in this exercise), persists across users, and is updated only when the underlying KB documents change. Think of it as the product's documentation, lifted into a queryable form.
- The **conversation history** is short-term *dialogue* memory. Per session, in-memory, no schema. Module 05 was about how to keep it from blowing up the context window.
- The **episodic memory** is structured *user-state* memory. Per session in this exercise, but the JSON dump at the end is everything you need to persist it across sessions; on the next visit the agent would load the dump back into `memory` before turn 1.

The three are designed not to overlap. Putting issue tracking inside the conversation history would force the agent to re-read every turn to find out what the user is working on; putting product documentation inside the episodic memory would force re-indexing every time a user logs in. Each piece lives where it can be queried cheaply.

### RAG and CAG together, not against each other

The slide deck `08 RAG vs CAG` frames the two as alternatives, but in practice they compose. RAG handles the *specific* questions (pricing for Business plan, SLA on Starter, supported integrations), CAG handles everything that falls outside the corpus's footprint (general orientation, how to reach support, compliance posture, off-topic small talk). The score-threshold gate is the routing rule between them. The pattern generalises: any time a retrieval system has a non-trivial probability of returning irrelevant results, a CAG-style fallback turns silent failures into graceful degradations.

On this KB and embedder, on-topic queries cluster around 0.55-0.70 in relevance score and off-topic ones around 0.15-0.30, wide enough that 0.4 separates them cleanly. The actual run also revealed that *the KB has broader coverage than I anticipated*: my pre-run prediction was 4 RAG / 4 CAG over the ten test turns, but the run came in at 8 RAG / 2 CAG. Turns I expected to fall through to CAG (e.g. "what is the average response time on Friday afternoons") matched the SLA chunk with 0.62 because SLA covers business hours explicitly. That is not a bug, it is the retrieval doing its job better than I gave it credit for. When the corpus genuinely covers more ground than expected, the score gate is the right place to set the *minimum* useful similarity, not a hard partition.

### What the first run revealed about structured extraction

The episodic-memory extractor went through three iterations before it produced correct outputs on the 10-turn dialogue:

1. **Iteration 1** (`Optional[str]` fields, terse descriptions). Every field came back as `null`. With Optional fields, the schema gives the model an easy escape route: when in doubt, return null.
2. **Iteration 2** (`Optional[str]`, explicit decision rules in the prompt). The `plan` field came back as sentence-long summaries of the chosen plan ("Business plan for up to 20 users including all features..."), and `new_issue` / `resolved_issue` still came back null even when the user explicitly described a problem.
3. **Iteration 3** (`Literal["Starter", "Business", "Enterprise", "NONE"]` for plan, **required `str` with `"NONE"` sentinel** for issue fields, **four-shot examples** in the system prompt). All four fields landed correctly: the install issue was captured at turn 3, persisted in `open_issues` for five turns, and migrated to `resolved_issues` at turn 8 via the substring match. The plan came in as `"Business"` cleanly at turn 10.

Generalising from these three iterations: **prompt-level constraints on the output shape are unreliable on small chat models**. Three patterns that actually work:

- Typed schemas with `Literal` enums (the model is forced to pick from a small set).
- Required fields with explicit "no update" sentinel values (`"NONE"` rather than `null`), which forces the model to make a per-turn decision instead of defaulting to absent.
- Few-shot examples in the prompt showing the exact input/output mapping for each canonical case.

Optional fields combined with abstract instructions look like the most natural design, but they put too much load on the model's judgment about whether to fill a field. The robust path is to constrain the schema first and let the prompt explain rather than enforce.

### Soft adherence of the proactive trigger

The `check_persistent_issues` function correctly inserts a `PROACTIVE INTERVENTION` instruction into the system prompt at turns 6 and 7 (when the install issue has been open for three and four turns, respectively). The instruction tells the assistant to suggest contacting `support@proflow.io` for direct help. In the final run the trigger fired in the prompt as designed, but the model chose to answer the user's immediate question (Jira import, SLA times) without appending the support reminder. The trigger is present, the model's adherence to it is soft.

This is a different kind of failure from the extractor bugs above: not a typing or schema problem, but a *prompt-priority* problem. The model has two competing instructions ("answer the user's question concisely" and "suggest contacting support"), and on `qwen2.5:14b` the immediate-question instruction wins. Two ways to harden this for a real deployment:

- **Make the trigger directive rather than suggestive** in the system prompt: change *"Suggest the user contact support@proflow.io"* to *"You MUST end your response with: 'For direct help on this open issue, please contact support@proflow.io.'"*. Stronger phrasing raises the probability without guaranteeing it.
- **Move the trigger out of the prompt entirely** and into post-processing. After the response is generated, if `check_persistent_issues` returned a non-empty string for this turn, append a fixed reminder sentence to the response. 100% adherence, no extra LLM call, at the cost of a slightly more rigid output.

For this exercise I chose to leave the soft-fail visible: it is the same lesson as the source-tag inconsistency (which we fixed with post-processing), and it makes the broader principle concrete - prompt-level instructions are best-effort, post-processing is guaranteed.

### What I would change for a real deployment

Three concrete moves that this exercise does not make but production would.

First, **persistent vector storage**. `Chroma.from_documents` without a `persist_directory` keeps the index in memory only; the next run rebuilds it. For a real KB of hundreds of documents that is a multi-second startup cost per process and a load on the embedder. One constructor argument fixes it.

Second, **episodic memory persisted to a key-value store**, keyed by user ID. Redis or sqlite is enough for thousands of users; once the schema grows (entity memory: track every product, person, or project the user mentions, not just issues) a small relational schema starts to pay off.

Third, **reranking on top of retrieval**. The current pipeline takes whatever the vector store returns. A cross-encoder reranker over the top-10 candidates would push relevance higher and let me lower the score threshold (catching more nuanced queries) without inflating the false-positive rate. The cost is one extra model call per turn, which on a hosted embedder is negligible.

### What this is still not solving

Three known limitations worth flagging.

The KB is in English; queries are accepted in English only. A multilingual KB would need either translation of the query at retrieval time, or per-language indexing, or a multilingual embedder. The PRJ in module 02 chose the third path with `paraphrase-multilingual-MiniLM`.

The episodic memory extractor runs every turn and treats each exchange independently. A user who rephrases the same issue twice ("the install is broken" then "installation still does not work") might end up with two separate entries in `open_issues`. Deduplication via embedding similarity on the issue descriptions would fix it, at the cost of an extra embedding call per memory update.

The proactive trigger fires on a hard threshold of three turns. In practice the right trigger is task-specific - some issues should escalate immediately (login broken, data loss), others can sit for days (cosmetic preferences). The threshold is the simplest possible policy; a richer one would weight by issue category.

### Forward link to module 07

Module 07 (*Deploy di un agente*) is where the in-memory artefacts above become real services. The vector store moves to a managed instance, the chat model behind an API gateway, the episodic memory into a real database, the agent behind an HTTP endpoint with auth and rate limiting. Most of the design work has already happened here: the boundaries between the three memories are clean, and each one has an obvious external counterpart in a production stack.